# TODO

- Start training and observe that the geminis recommendations (weight tying, swiglu) can improve training performance, memory and stability.

- Switch to multihead attention
- Understand swiglu implementation
- Use RMSNorm instead of layer Norm in the MLP block

- Test with this arch, benchmark regular attention vs flash attention (profile)
- Once it's done replace MLP block with Mixture of experts

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import torch
import torch.nn.functional as F
from nanomoe.data import helpers as data_helpers

In [2]:
# Finding out tokenizer min and max value to know type in which to encode tokens

# Vocab Size (V),Recommended PyTorch/Numpy Type,Max Value Allowed
# V<256,uint8,255
# "V<65,536",uint16,"65,535"
# "V>65,536",int32,~2.1 Billion

import tiktoken

enc = tiktoken.get_encoding("gpt2")
print("Max token value", enc.max_token_value) # -> uint16 !
print("Vocab size : ", enc.n_vocab)

Max token value 50256
Vocab size :  50257


In [3]:
# Test toy data loading

input_file_path = os.path.join("..","data","toy_data.txt")
output_path = os.path.join("..","data")

data_helpers.dload_toy_data(input_file_path=input_file_path, output_path=output_path)

train has 301,966 tokens
val has 36,059 tokens


In [4]:
# Test dataset
from torch.utils.data import DataLoader
from nanomoe.data import loading

file_path = os.path.join("..","data","train.bin")
block_size = 32
batch_size = 10

ds = loading.ShakespearDs(path = file_path, block_size = block_size)
loader = DataLoader(
    ds,
    batch_size = batch_size,
    shuffle=True,
    pin_memory=True, # Not ideal for mac -> useful in prod env
    num_workers=2, # Not ideal for mac -> useful in prod env
    prefetch_factor=2
)

for X,y in loader:
    break

/Users/sevan/code/projects/research_engineer/NanoMoE/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
from nanomoe import model

vocab_size = 50257
n_embed = 8

transf = model.GPT(vocab_size=vocab_size,n_embed=n_embed, attention="standard", block_size=block_size)

out = transf(X)

lfun = torch.nn.CrossEntropyLoss()
loss = F.cross_entropy(out.permute(0,2,1),y.to(torch.long))
loss

tensor(11.2092, grad_fn=<NllLoss2DBackward0>)

In [26]:
B,C,T = out.shape
out.view(B,T,C).shape

torch.Size([10, 50257, 32])

In [ ]:
import os
import tiktoken
from torch.utils.data import DataLoader
from nanomoe.data import loading
from nanomoe import model, trainer

enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab

block_size = 32
batch_size = 10
n_embed = 8

batch = (torch.rand(batch_size,block_size).to(torch.long), torch.rand(batch_size,block_size).to(torch.long))

gpt = model.GPT(
    vocab_size=vocab_size,
    n_embed=n_embed,
    attention="standard",
    block_size=block_size
)
lgpt = trainer.LightningGPT(model=gpt)


# try :
#     lgpt.training_step(batch=batch, batch_idx=0)
# except Exception as e:
#     raise Exception(e)

In [27]:
import yaml

with open("../configs/train_mac.yaml", "r") as f:
    test = yaml.safe_load(f)
test["loader"]

{'batch_size': 16,
 'shuffle': True,
 'pin_memory': False,
 'num_workers': 0,
 'prefetch_factor': 2}

In [31]:
from nanomoe import config

model_config = config.GPTConfig.from_yaml("../configs/gpt.yaml")
experiment_config = config.ExperimentConfig.from_yaml("../configs/train_mac.yaml")

In [33]:
experiment_config.model_dump()

{'loader': {'batch_size': 16,
  'shuffle': True,
  'pin_memory': False,
  'num_workers': 0,
  'prefetch_factor': 2},
 'logger': {'type': 'wandb',
  'project': 'nanomoe',
  'log_model': False,
  'log_every_n_steps': 1,
  'max_epochs': 5}}

In [13]:
import yaml

test = yaml.safe_load("../configs/gpt.yaml")
test

'../configs/gpt.yaml'

In [10]:
import lightning as L
from lightning.pytorch.loggers import WandbLogger

file_path = os.path.join("..","data","train.bin")
block_size = 32
batch_size = 10

ds = loading.ShakespearDs(path = file_path, block_size = block_size)
train_loader = DataLoader(
    ds,
    batch_size = batch_size,
    shuffle=True,
    pin_memory=True, # Not ideal for mac -> useful in prod env
    num_workers=2, # Not ideal for mac -> useful in prod env
    prefetch_factor=2
)

wandb_logger = WandbLogger(project="nanomoe", log_model=False)

trainer = L.Trainer(
    logger=wandb_logger,
    log_every_n_steps=1,
    overfit_batches=1,
    max_epochs=5
)
trainer.fit(lgpt, train_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name  | Type | Params | Mode  | FLOPs
-----------------------------------------------
0 | model | GPT  | 855 K  | train | 0    
-----------------------------------------------
855 K     Trainable params
0         Non-trainable params
855 K     Total params
3.422     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 4: 100%|██████████| 1/1 [00:06<00:00,  0.15it/s, v_num=zhon, train_loss=11.10]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 1/1 [00:06<00:00,  0.15it/s, v_num=zhon, train_loss=11.10]


In [ ]:
import torch
x = torch.rand(batch_size,block_size)
y = torch.rand(batch_size,block_size)

In [11]:
y.shape

torch.Size([10, 32])